# DOT to CSCL mapping

This project is about NYC street-level roadwork. The basic question is whether weather, plow activity, traffic, and earlier roadwork help explain or predict later resurfacing.

The resurfacing file used here comes from DOT, meaning the New York City Department of Transportation. It has project-level roadwork records, but the street names in it are not in the same format as the street reference file used in the rest of the project.

CSCL is the Citywide Street Centerline file. It is the street reference table used here to give street names a more consistent form. The point of this notebook is to map the DOT street names onto CSCL street names so the resurfacing data can be joined to the rest of the street-level data later.

Output: `data/derived/dot_to_cscl.csv`.


In [1]:
import pandas as pd

In [2]:
dot = pd.read_csv("data/raw/roadwork/dot_inhouse_resurfacing.csv")

/var/folders/zh/4snkqr657fb06hs2r74_4xpm0000gn/T/ipykernel_1655/3340044899.py:1: DtypeWarning: Columns (5,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  dot = pd.read_csv("data/raw/roadwork/dot_inhouse_resurfacing.csv")


In [3]:
dot.head(1000)

,OFT Code,Project Type,Borough Code,Location On Street,Location From Street,Location To Street,Project ID,Project Status,Project Speed Bumps,Location Community Board,...,Location Actual Milling End Date,Location Actual Paving Start Date,Location Actual Paving End Date,Location Actual Protect Until,Location Actual Lane Miles Paved,Location Actual Paving Square Yard,Location Status,Location Segment ID,Location WKT,Location Node ID
0,110010117010000000,Intersection,M,1 AVENUE,EAST 1 STREET,NaN,M2020-03-09,Completed Project,NaN,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20780.0,POINT (987456.88670000434 202779.81419999897),NaN
1,110010117010000000,Intersection,M,1 AVENUE,EAST 1 STREET,NaN,M2009-03-06,Completed Project,0.0,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20780.0,POINT (987456.88670000434 202779.81419999897),NaN
2,110010117030000000,Intersection,M,1 AVENUE,EAST 2 STREET,NaN,M2009-03-09,Completed Project,0.0,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20787.0,POINT (987584.33400000632 203018.48489999771),NaN
3,110010117030000000,Intersection,M,1 AVENUE,EAST 2 STREET,NaN,M2021-03-24,Paving Completed,NaN,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20787.0,POINT (987584.33400000632 203018.48489999771),NaN
4,110010117030000000,Intersection,M,1 AVENUE,EAST 2 STREET,NaN,M2016-03-03,Completed Project,0.0,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20787.0,POINT (987584.33400000632 203018.48489999771),NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,110610131190000000,Intersection,M,7 AVENUE,ST NICHOLAS AVENUE,NaN,M2002-10-2,Completed Project,NaN,110.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24140.0,MULTIPOINT ((997392.21089999378 231914.0204000...,NaN
996,110610131190000000,Intersection,M,7 AVENUE,ST NICHOLAS AVENUE,NaN,M2002-10-2,Completed Project,NaN,110.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24141.0,MULTIPOINT ((997392.21089999378 231914.0204000...,NaN
997,110610131190000000,Intersection,M,7 AVENUE,ST NICHOLAS AVENUE,NaN,M2002-10-2,Completed Project,NaN,110.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78498.0,MULTIPOINT ((997392.21089999378 231914.0204000...,NaN
998,110610134010000000,Intersection,M,7 AVENUE,WEST 12 STREET,NaN,M2017-02-08,Completed Project,0.0,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20549.0,POINT (984070.62690000236 207869.19709999859),NaN


In [4]:
dot = dot[["OFT Code", "Borough Code", "Location On Street"]]

In [5]:
dot.shape

(601356, 3)

In [6]:
cscl = pd.read_csv("data/reference/CSCL.csv")

In [7]:
cscl[cscl["PHYSICALID"] == 3]

,the_geom,PHYSICALID,L_LOW_HN,L_HIGH_HN,R_LOW_HN,R_HIGH_HN,L_ZIP,R_ZIP,STATUS,BIKE_LANE,...,Post Directional,Post Modifier,Full Street Name,BIKE TRAFFIC DIRECTION,SHAPE__Length,GlobalID,SEGMENT_TYPE,SEGMENT_TYPE_VALUE,STREET NAME,Street Name Label
98772,MULTILINESTRING ((-74.017931916875 40.70618310...,3,50,64,51,63,10280.0,10280.0,2,NaN,...,NaN,NaN,BATTERY PL,NaN,105.855292,d01cd096-9198-4992-b415-a7b8e2a23135,NaN,NaN,BATTERY,BATTERY PL


In [8]:
cscl = cscl[["Borough Code", "Full Street Name", "STREET NAME", "Street Name Label", "PHYSICALID"]]

In [9]:
cscl.head()

,Borough Code,Full Street Name,STREET NAME,Street Name Label,PHYSICALID
0,3,AVE N,N,AVE N,46810
1,2,HONE AVE,HONE,HONE AVE,86757
2,4,48 ST,48,48 ST,84282
3,1,LAIGHT ST,LAIGHT,LAIGHT ST,79741
4,1,W 60 ST,60,W 60 ST,191409


In [10]:
# Borough code used in DOT df mapping to CSCL borough code

borough_dct = {'M':1, 'X':2, 'B':3, 'Q':4, 'S':5}

In [11]:
import pandas as pd
from street_normalize import normalize

# --- Borough code mapping ---
borough_dct = {'M': '1', 'X': '2', 'B': '3', 'Q': '4', 'S': '5'}

# --- Preprocess DOT dataset ---
dot_prep = (
    dot[['Borough Code', 'Location On Street']]
    .dropna()
    .assign(**{
        'Borough Code Num': lambda df: df['Borough Code'].map(borough_dct),
        'Location On Street': lambda df: df['Location On Street'].str.lower()
    })
    .drop_duplicates(['Borough Code Num', 'Location On Street'])
)
dot_prep['norm'] = dot_prep['Location On Street'].map(normalize)

# --- Preprocess CSCL dataset ---
cscl_prep = cscl.copy()
cscl_prep['Borough Code'] = cscl_prep['Borough Code'].astype(str)
cscl_prep['Full Street Name'] = cscl_prep['Full Street Name'].str.lower()
cscl_prep['norm'] = cscl_prep['Full Street Name'].map(normalize)

# --- Merge on borough + normalized name ---
# --- Merge on borough + normalized name, include PHYSICALID ---
merged = dot_prep.merge(
    cscl_prep[['Borough Code', 'Full Street Name', 'PHYSICALID', 'norm']],
    left_on=['Borough Code Num', 'norm'],
    right_on=['Borough Code', 'norm'],
    how='left',
    suffixes=('_dot', '_cscl')
)


print(merged.columns.tolist())



['Borough Code_dot', 'Location On Street', 'Borough Code Num', 'norm', 'Borough Code_cscl', 'Full Street Name', 'PHYSICALID']


In [12]:
# --- Result DataFrame ---
# --- Select and rename columns for final result ---
result = merged[['Borough Code Num', 'Location On Street', 'Full Street Name', 'PHYSICALID']].rename(
    columns={
        'Borough Code Num': 'Borough Code',
        'Location On Street': 'DOT street name',
        'Full Street Name': 'CSCL street name'
    }
).drop_duplicates()

result['PHYSICALID'] = result['PHYSICALID'].apply(lambda x: int(x) if pd.notna(x) else x)

# --- Optional: mapping dictionary ---
mapping = (
    merged.groupby(['Borough Code Num', 'Location On Street'])['Full Street Name']
    .apply(lambda x: set(x.dropna()))
    .to_dict()
)


In [13]:
result

,Borough Code,DOT street name,CSCL street name,PHYSICALID
0,1,1 avenue,1 ave,2696.0
1,1,1 avenue,1 ave,140711.0
2,1,1 avenue,1 ave,2607.0
3,1,1 avenue,1 ave,2679.0
4,1,1 avenue,1 ave,2664.0
...,...,...,...,...
101677,1,cpw 100 approach,NaN,NaN
101678,4,mars place,mars pl,81404.0
101679,4,vaswani avenue,vaswani ave,184428.0
101680,5,fremont street,fremont st,168784.0


In [14]:
result.columns

Index(['Borough Code', 'DOT street name', 'CSCL street name', 'PHYSICALID'], dtype='object')

In [15]:
# remove Missing value (always substring)
# check that mapping is 1-to-1
# see condensed list of map after this verification

In [16]:
missing_cscl = result[result['CSCL street name'].isna()]

In [17]:
# Drop rows where CSCL street name is missing
result_clean = result.dropna(subset=['CSCL street name'])

# Save to CSV
result_clean.to_csv('data/derived/dot_to_cscl.csv', index=False)

In [18]:
missing_cscl

,Borough Code,DOT street name,CSCL street name,PHYSICALID
696,5,bend,NaN,NaN
1416,1,miller hy et nb,NaN,NaN
1417,1,miller hy en sb,NaN,NaN
2451,1,bend,NaN,NaN
2496,1,west 110 street,NaN,NaN
...,...,...,...,...
101612,4,van wyck expressway exit 9 nb,NaN,NaN
101613,1,nj transit railroad,NaN,NaN
101666,1,cp lenox approach,NaN,NaN
101671,1,cpw 110 approach,NaN,NaN


Processing to fill in missing CSCL street names. Canonical raw inputs now live under `data/raw/roadwork/` and `data/reference/`.

In [19]:
# look at what corresponds to given DOT name

dot_name = "union"

# Lowercase and split into words
dot_words = dot_name.lower().split()

# Filter CSCL rows by borough first, if desired
cscl_candidates = cscl_prep[cscl_prep['Borough Code'] == '2']  # replace '1' with borough code

# Check if all words appear as substrings in CSCL street
def contains_all_words(cscl_street, words):
    cscl_street_lower = cscl_street.lower()
    return all(w in cscl_street_lower for w in words)

matches = cscl_candidates[cscl_candidates['Full Street Name'].apply(lambda x: contains_all_words(x, dot_words))]

matches[['Full Street Name', 'PHYSICALID']]

,Full Street Name,PHYSICALID
585,union ave,149221
1485,unionport rd,41673
3396,unionport rd,41681
5355,unionport brg,41536
11653,unionport rd,121033
...,...,...
110800,union ave,193021
111122,unionport rd,41689
113129,unionport rd,117691
114218,unionport rd,185473


In [20]:
s = cscl[cscl["PHYSICALID"] == 87765]["Full Street Name"].values[0]
s.lower() == 'andrews ave'
normalize('andrews ave')

s = dot[dot["Location On Street"] == "ANDREWS AVENUE"]["Location On Street"].values[0]
s.split(' ')

['ANDREWS', 'AVENUE']

In [21]:
# manually assign CSCL street name to given DOT street name. PHYSICALID will be assigned randomly based 
# on existing CSCL street name columns

manual_mapping = {
    '6 ave': 'ave of the americas',
    'avenue of the americas': 'ave of the americas',
    'west 110 street': '110 st',
    'andrews avenue north': 'andrews avenue',
    'andrews avenue south': 'andrews avenue'
}

# Filter rows that need manual fill
for dot_street, cscl_street in manual_mapping.items():
    # Get candidate PHYSICALIDs from CSCL
    candidates = cscl_prep[cscl_prep['Full Street Name'] == cscl_street]['PHYSICALID']
    
    if len(candidates) == 0:
        print(f"No CSCL entry found for {cscl_street}")
        continue
    
    chosen_physicalid = candidates.sample(1).iloc[0]  # randomly pick one PHYSICALID
    
    # Update the result df
    result.loc[result['DOT street name'] == dot_street, 'CSCL street name'] = cscl_street
    result.loc[result['DOT street name'] == dot_street, 'PHYSICALID'] = chosen_physicalid


No CSCL entry found for andrews avenue
No CSCL entry found for andrews avenue


In [22]:
# remove rows from result with junk DOT names

to_remove = ['bend', 'dead end', 'edens alley', 'mc kenna square', 'ramp', 'dr m l king jr boulevard', 'bissel avenue',
             'union square'] + [f"beach {i} street" for i in range(200)]
result = result[~result['DOT street name'].isin(to_remove)]